# Notebook 2: `02_dimension_etl.ipynb`

## 1. Introduction

**Objective**: Extract raw Olist datasets, perform necessary transformations, and incrementally load them into the MySQL dimensional tables.

**Source Datasets**:
- `geolocation`
- `customers`
- `sellers`
- `products`
- `category translation`
- `orders`

**Target Tables**:
- `dim_geography`
- `dim_customer`
- `dim_seller`
- `dim_product`
- `dim_date`

**ETL Workflow**: We will implement one dimension at a time, strictly following an Extract -> Transform -> Validate (Polars) -> Load (MySQL) -> Validate (SQL) lifecycle.


## 2. Imports & Configuration
Load necessary libraries and environment variables.

In [1]:
import polars as pl
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os
import logging
from pathlib import Path

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Load environment variables
load_dotenv(Path("../.env"))

DATA_PATH = Path("../data/raw")

## 3. Database Connection
Create the SQLAlchemy engine and test the connection to MySQL.

In [2]:
# Build MySQL Connection String
import os
import urllib.parse
from sqlalchemy import create_engine, text
import logging

user = os.environ.get("DB_USER")

# URL-encode the password to safely handle special characters like '@'
raw_password = os.environ.get("DB_PASSWORD", "")
password = urllib.parse.quote_plus(raw_password)

host = os.environ.get("DB_HOST", "localhost")
port = os.environ.get("DB_PORT", "3306")
database = os.environ.get("DB_NAME", "brazilian_ecommerce_dw")

conn_str = f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
engine = create_engine(conn_str)

# Test Connection
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT DATABASE();"))
        db_name = result.scalar()
        logging.info(f"Successfully connected to database: {db_name}")
except Exception as e:
    logging.error(f"Failed to connect: {e}")

2026-07-26 02:10:32,705 - INFO - Successfully connected to database: brazilian_ecommerce_dw


## 3.1 Reset Data Warehouse (Development Only)
Run this cell if you need to wipe all data and start the pipeline from scratch. It safely deletes records in **reverse dependency order** to avoid `Foreign Key` constraint errors.

In [3]:
logging.info("Wiping existing warehouse data in reverse dependency order...")

with engine.begin() as conn:
    # We must delete in reverse dependency order due to ON DELETE RESTRICT constraints
    # conn.execute(text("DELETE FROM fact_sales;")) # Not created yet, uncomment later
    conn.execute(text("DELETE FROM dim_customer;"))
    conn.execute(text("DELETE FROM dim_seller;"))
    conn.execute(text("DELETE FROM dim_product;"))
    conn.execute(text("DELETE FROM dim_geography;"))
    conn.execute(text("DELETE FROM dim_date;"))

logging.info("Warehouse successfully reset! Ready for fresh ETL.")

2026-07-26 02:10:32,714 - INFO - Wiping existing warehouse data in reverse dependency order...


2026-07-26 02:11:28,179 - INFO - Warehouse successfully reset! Ready for fresh ETL.


## 4. Load Source Data
Load all the raw Olist datasets into memory via Polars once to optimize I/O.

In [4]:
logging.info("Loading raw datasets...")

geolocation = pl.read_csv(DATA_PATH / "olist_geolocation_dataset.csv")
customers = pl.read_csv(DATA_PATH / "olist_customers_dataset.csv")
sellers = pl.read_csv(DATA_PATH / "olist_sellers_dataset.csv")
products = pl.read_csv(DATA_PATH / "olist_products_dataset.csv")
categories = pl.read_csv(DATA_PATH / "product_category_name_translation.csv")
orders = pl.read_csv(DATA_PATH / "olist_orders_dataset.csv")

logging.info("Source data loaded successfully.")

2026-07-26 02:11:28,311 - INFO - Loading raw datasets...


2026-07-26 02:11:28,966 - INFO - Source data loaded successfully.


## 5. Geography Dimension ETL
**Base Dimension**: We construct this entirely from the `geolocation` dataset to maintain geographic authority.

**Transformations**:
- Remove duplicate locations
- Average latitude and longitude for the same ZIP code


In [5]:
# EXTRACT is already handled (geolocation loaded)

# TRANSFORM
logging.info("Transforming Geography Dimension...")

# We group purely by zip_code_prefix to avoid MySQL collation collisions
# (e.g. 'são paulo' vs 'sao paulo' which MySQL treats as duplicates under unicode_ci)
dim_geography_df = (
    geolocation
    .rename({
        "geolocation_zip_code_prefix": "zip_code_prefix",
        "geolocation_city": "city_name",
        "geolocation_state": "state_code",
        "geolocation_lat": "latitude",
        "geolocation_lng": "longitude"
    })
    .group_by("zip_code_prefix")
    .agg([
        pl.col("city_name").first(),
        pl.col("state_code").first(),
        pl.col("latitude").mean(),
        pl.col("longitude").mean()
    ])
)

2026-07-26 02:11:28,978 - INFO - Transforming Geography Dimension...


In [6]:
# VALIDATE (Polars)
# Assertions
assert dim_geography_df["zip_code_prefix"].n_unique() == dim_geography_df.height, "Duplicate zip codes found!"
assert dim_geography_df.null_count().sum_horizontal()[0] == 0, "Null values found!"

print("Shape:", dim_geography_df.shape)
print("\nSample:")
print(dim_geography_df.head(3))
logging.info("Geography validation passed.")

2026-07-26 02:11:29,143 - INFO - Geography validation passed.


Shape: (19015, 5)

Sample:
shape: (3, 5)
┌─────────────────┬───────────────┬────────────┬────────────┬────────────┐
│ zip_code_prefix ┆ city_name     ┆ state_code ┆ latitude   ┆ longitude  │
│ ---             ┆ ---           ┆ ---        ┆ ---        ┆ ---        │
│ i64             ┆ str           ┆ str        ┆ f64        ┆ f64        │
╞═════════════════╪═══════════════╪════════════╪════════════╪════════════╡
│ 85868           ┆ foz do iguacu ┆ PR         ┆ -25.487219 ┆ -54.576317 │
│ 1316            ┆ sao paulo     ┆ SP         ┆ -23.552955 ┆ -46.639433 │
│ 45880           ┆ camacan       ┆ BA         ┆ -15.41435  ┆ -39.495594 │
└─────────────────┴───────────────┴────────────┴────────────┴────────────┘


Load into MySQL using SQLAlchemy, then Validate via SQL.

In [7]:
%%time
# LOAD
logging.info("Loading Geography Dimension to MySQL...")

df_pandas = dim_geography_df.to_pandas()

with engine.begin() as conn:
    # Use DELETE instead of TRUNCATE to respect foreign key constraints
    # (This will fail safely if dependent rows exist in customer/seller)
    conn.execute(text("DELETE FROM dim_geography;"))
    
    # Load data
    df_pandas.to_sql("dim_geography", conn, if_exists="append", index=False)
    
logging.info(f"Loaded {len(df_pandas):,} records into dim_geography.")

2026-07-26 02:11:29,199 - INFO - Loading Geography Dimension to MySQL...


2026-07-26 02:11:30,508 - INFO - Note: NumExpr detected 22 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.


2026-07-26 02:11:30,509 - INFO - NumExpr defaulting to 16 threads.


2026-07-26 02:11:37,781 - INFO - Loaded 19,015 records into dim_geography.


CPU times: total: 875 ms
Wall time: 8.58 s


In [8]:
# VALIDATE (SQL)
with engine.connect() as conn:
    row_count = conn.execute(text("SELECT COUNT(*) FROM dim_geography;")).scalar()
    print(f"Total Rows in MySQL dim_geography: {row_count}")
    
    sample = conn.execute(text("SELECT * FROM dim_geography LIMIT 3;")).fetchall()
    print("\nSample Database Records:")
    for row in sample:
        print(row)

Total Rows in MySQL dim_geography: 19015

Sample Database Records:
(228181, 85868, 'foz do iguacu', 'PR', Decimal('-25.48721939'), Decimal('-54.57631711'), datetime.datetime(2026, 7, 26, 2, 11, 32), datetime.datetime(2026, 7, 26, 2, 11, 32))
(228182, 1316, 'sao paulo', 'SP', Decimal('-23.55295485'), Decimal('-46.63943282'), datetime.datetime(2026, 7, 26, 2, 11, 32), datetime.datetime(2026, 7, 26, 2, 11, 32))
(228183, 45880, 'camacan', 'BA', Decimal('-15.41435013'), Decimal('-39.49559371'), datetime.datetime(2026, 7, 26, 2, 11, 32), datetime.datetime(2026, 7, 26, 2, 11, 32))


## 6. Customer Dimension ETL
**Transformations**:
- Read `dim_geography` from MySQL to retrieve the generated `geography_key`
- Join `customers` with `dim_geography` on `zip_code_prefix`
- Select required columns and validate integrity


In [9]:
%%time
# TRANSFORM
logging.info("Transforming Customer Dimension...")

# 1. Fetch generated geography_keys from MySQL
query = "SELECT geography_key, zip_code_prefix FROM dim_geography;"
dim_geography_mysql = pl.read_database(query, engine.connect())

# 2. Join customers with geography keys
dim_customer_df = (
    customers
    .join(
        dim_geography_mysql,
        left_on="customer_zip_code_prefix",
        right_on="zip_code_prefix",
        how="left"
    )
    .select([
        pl.col("customer_id"),
        pl.col("customer_unique_id"),
        pl.col("geography_key")
    ])
)

2026-07-26 02:11:37,826 - INFO - Transforming Customer Dimension...


CPU times: total: 203 ms
Wall time: 499 ms


In [10]:
# VALIDATE (Polars) & METRICS
# Check row count hasn't multiplied or dropped
assert dim_customer_df.height == customers.height, "Row count mismatch after join!"
# Check business key uniqueness
assert dim_customer_df["customer_id"].n_unique() == dim_customer_df.height, "Duplicate customer_ids found!"

# Check FK completeness & Investigate Orphans
orphans = dim_customer_df.filter(pl.col("geography_key").is_null())
null_keys = orphans.height

if null_keys > 0:
    logging.warning(f"Found {null_keys} orphan customers without a geography_key! They will be loaded as NULL.")
    
    # Investigate Orphan Zip Codes
    orphan_zips = customers.join(orphans.select("customer_id"), on="customer_id", how="semi").select([
        "customer_zip_code_prefix", 
        "customer_city", 
        "customer_state"
    ]).unique().sort("customer_state")
    
    print("\n--- ORPHAN INVESTIGATION ---")
    print(f"Total Unique Orphan ZIPs: {orphan_zips.height}")
    print(orphan_zips.head(10))

# ETL Metrics
matched = dim_customer_df.height - null_keys
success_rate = (matched / dim_customer_df.height) * 100

print("\n--- ETL METRICS ---")
print(f"Rows processed:    {customers.height:,}")
print(f"Rows loaded:       {dim_customer_df.height:,}")
print(f"Matched geography: {matched:,}")
print(f"Missing geography: {null_keys:,}")
print(f"Success rate:      {success_rate:.2f}%")

print("\nShape:", dim_customer_df.shape)
logging.info("Customer validation passed.")

2026-07-26 02:11:38,336 - WARNING - Found 278 orphan customers without a geography_key! They will be loaded as NULL.


2026-07-26 02:11:38,349 - INFO - Customer validation passed.



--- ORPHAN INVESTIGATION ---
Total Unique Orphan ZIPs: 157
shape: (10, 3)
┌──────────────────────────┬──────────────────┬────────────────┐
│ customer_zip_code_prefix ┆ customer_city    ┆ customer_state │
│ ---                      ┆ ---              ┆ ---            │
│ i64                      ┆ str              ┆ str            │
╞══════════════════════════╪══════════════════╪════════════════╡
│ 57254                    ┆ luziapolis       ┆ AL             │
│ 42843                    ┆ jaua             ┆ BA             │
│ 44135                    ┆ humildes         ┆ BA             │
│ 42716                    ┆ lauro de freitas ┆ BA             │
│ 41347                    ┆ salvador         ┆ BA             │
│ 43870                    ┆ jacuipe          ┆ BA             │
│ 48504                    ┆ aribice          ┆ BA             │
│ 41098                    ┆ salvador         ┆ BA             │
│ 61906                    ┆ maracanau        ┆ CE             │
│ 62898        

In [11]:
%%time
# LOAD
logging.info("Loading Customer Dimension to MySQL...")

df_pandas = dim_customer_df.to_pandas()

with engine.begin() as conn:
    conn.execute(text("DELETE FROM dim_customer;"))
    df_pandas.to_sql("dim_customer", conn, if_exists="append", index=False)

logging.info(f"Loaded {len(df_pandas):,} records into dim_customer.")

2026-07-26 02:11:38,354 - INFO - Loading Customer Dimension to MySQL...


2026-07-26 02:12:56,029 - INFO - Loaded 99,441 records into dim_customer.


CPU times: total: 984 ms
Wall time: 1min 17s


In [12]:
# VALIDATE (SQL)
with engine.connect() as conn:
    row_count = conn.execute(text("SELECT COUNT(*) FROM dim_customer;")).scalar()
    print(f"Total Rows in MySQL dim_customer: {row_count}")
    
    sample = conn.execute(text("SELECT * FROM dim_customer LIMIT 3;")).fetchall()
    print("\nSample Database Records:")
    for row in sample:
        print(row)

Total Rows in MySQL dim_customer: 99441



Sample Database Records:
(807721, '06b8999e2fba1a1fbc88172c00ba8bc7', '861eff4711a542e4b93843c6dd7febb0', 241064, datetime.datetime(2026, 7, 26, 2, 11, 39), datetime.datetime(2026, 7, 26, 2, 11, 39))
(807722, '18955e83d337fd6b2def6b18a428ac77', '290c77bc529b7ac935b93aa66c333dc3', 232466, datetime.datetime(2026, 7, 26, 2, 11, 39), datetime.datetime(2026, 7, 26, 2, 11, 39))
(807723, '4e7b3e00288586ebd08712fdd0374a03', '060e732b5b29e8181a18229c7b0b2b5e', 239508, datetime.datetime(2026, 7, 26, 2, 11, 39), datetime.datetime(2026, 7, 26, 2, 11, 39))


## 7. Seller Dimension ETL
**Transformations**:
- Read `dim_geography` from MySQL to retrieve the generated `geography_key`
- Join `sellers` with `dim_geography` on `zip_code_prefix`
- Select required columns and validate integrity


In [13]:
%%time
# TRANSFORM
logging.info("Transforming Seller Dimension...")

# 1. Fetch generated geography_keys from MySQL
query = "SELECT geography_key, zip_code_prefix FROM dim_geography;"
dim_geography_mysql = pl.read_database(query, engine.connect())

# 2. Join sellers with geography keys
dim_seller_df = (
    sellers
    .join(
        dim_geography_mysql,
        left_on="seller_zip_code_prefix",
        right_on="zip_code_prefix",
        how="left"
    )
    .select([
        pl.col("seller_id"),
        pl.col("geography_key")
    ])
)

# VALIDATE (Polars) & METRICS
# Check row count hasn't multiplied or dropped
assert dim_seller_df.height == sellers.height, "Row count mismatch after join!"
# Check business key uniqueness
assert dim_seller_df["seller_id"].n_unique() == dim_seller_df.height, "Duplicate seller_ids found!"

# Check FK completeness & Investigate Orphans
orphans = dim_seller_df.filter(pl.col("geography_key").is_null())
null_keys = orphans.height

if null_keys > 0:
    logging.warning(f"Found {null_keys} orphan sellers without a geography_key! They will be loaded as NULL.")
    
    # Investigate Orphan Zip Codes
    orphan_zips = sellers.join(orphans.select("seller_id"), on="seller_id", how="semi").select([
        "seller_zip_code_prefix", 
        "seller_city", 
        "seller_state"
    ]).unique().sort("seller_state")
    
    print("\n--- ORPHAN INVESTIGATION ---")
    print(f"Total Unique Orphan ZIPs: {orphan_zips.height}")
    print(orphan_zips.head(10))

# ETL Metrics
matched = dim_seller_df.height - null_keys
success_rate = (matched / dim_seller_df.height) * 100

print("\n--- ETL METRICS ---")
print(f"Rows processed:    {sellers.height:,}")
print(f"Rows loaded:       {dim_seller_df.height:,}")
print(f"Matched geography: {matched:,}")
print(f"Missing geography: {null_keys:,}")
print(f"Success rate:      {success_rate:.2f}%")

print("\nShape:", dim_seller_df.shape)
logging.info("Seller validation passed.")

2026-07-26 02:12:56,339 - INFO - Transforming Seller Dimension...


2026-07-26 02:12:57,364 - WARNING - Found 7 orphan sellers without a geography_key! They will be loaded as NULL.


2026-07-26 02:12:57,380 - INFO - Seller validation passed.



--- ORPHAN INVESTIGATION ---
Total Unique Orphan ZIPs: 7
shape: (7, 3)
┌────────────────────────┬─────────────────┬──────────────┐
│ seller_zip_code_prefix ┆ seller_city     ┆ seller_state │
│ ---                    ┆ ---             ┆ ---          │
│ i64                    ┆ str             ┆ str          │
╞════════════════════════╪═════════════════╪══════════════╡
│ 71551                  ┆ brasilia        ┆ DF           │
│ 72580                  ┆ brasilia        ┆ DF           │
│ 37708                  ┆ pocos de caldas ┆ MG           │
│ 82040                  ┆ curitiba        ┆ PR           │
│ 91901                  ┆ porto alegre    ┆ RS           │
│ 2285                   ┆ sao paulo       ┆ SP           │
│ 7412                   ┆ aruja           ┆ SP           │
└────────────────────────┴─────────────────┴──────────────┘

--- ETL METRICS ---
Rows processed:    3,095
Rows loaded:       3,095
Matched geography: 3,088
Missing geography: 7
Success rate:      99.77%

Shap

In [14]:
%%time
# LOAD
logging.info("Loading Seller Dimension to MySQL...")

df_pandas = dim_seller_df.to_pandas()

with engine.begin() as conn:
    conn.execute(text("DELETE FROM dim_seller;"))
    df_pandas.to_sql("dim_seller", conn, if_exists="append", index=False)

logging.info(f"Loaded {len(df_pandas):,} records into dim_seller.")

2026-07-26 02:12:57,401 - INFO - Loading Seller Dimension to MySQL...


2026-07-26 02:12:59,272 - INFO - Loaded 3,095 records into dim_seller.


CPU times: total: 62.5 ms
Wall time: 1.87 s


In [15]:
# VALIDATE (SQL)
with engine.connect() as conn:
    row_count = conn.execute(text("SELECT COUNT(*) FROM dim_seller;")).scalar()
    print(f"Total Rows in MySQL dim_seller: {row_count}")
    
    sample = conn.execute(text("SELECT * FROM dim_seller LIMIT 3;")).fetchall()
    print("\nSample Database Records:")
    for row in sample:
        print(row)

Total Rows in MySQL dim_seller: 3095

Sample Database Records:
(18571, '3442f8959a84dea7ee197c632cb2df15', 230671, datetime.datetime(2026, 7, 26, 2, 12, 57), datetime.datetime(2026, 7, 26, 2, 12, 57))
(18572, 'd1b65fc7debc3361ea86b5f14c68d2e2', 245112, datetime.datetime(2026, 7, 26, 2, 12, 57), datetime.datetime(2026, 7, 26, 2, 12, 57))
(18573, 'ce3ad9de960102d0677a81f5d0bb7b2d', 246126, datetime.datetime(2026, 7, 26, 2, 12, 57), datetime.datetime(2026, 7, 26, 2, 12, 57))


## 8. Product Dimension ETL
**Transformations**:
- Join `products` with `categories` (category translation) on `product_category_name`
- Calculate `product_volume_cm3` = length × width × height
- Select and clean attributes


In [16]:
%%time
# TRANSFORM
logging.info("Transforming Product Dimension...")

# 1. Join products with category translations and compute volume
dim_product_df = (
    products
    .join(
        categories,
        on="product_category_name",
        how="left"
    )
    .with_columns(
        (pl.col("product_length_cm") * pl.col("product_height_cm") * pl.col("product_width_cm"))
        .alias("product_volume_cm3")
    )
    .select([
        pl.col("product_id"),
        pl.col("product_category_name"),
        pl.col("product_category_name_english"),
        pl.col("product_name_lenght").alias("product_name_length"),
        pl.col("product_description_lenght").alias("product_description_length"),
        pl.col("product_photos_qty"),
        pl.col("product_weight_g"),
        pl.col("product_length_cm"),
        pl.col("product_height_cm"),
        pl.col("product_width_cm"),
        pl.col("product_volume_cm3")
    ])
)

# VALIDATE (Polars) & METRICS
# Check row count hasn't multiplied or dropped
assert dim_product_df.height == products.height, "Row count mismatch after join!"
# Check business key uniqueness
assert dim_product_df["product_id"].n_unique() == dim_product_df.height, "Duplicate product_ids found!"

# Check translation completeness & Investigate Orphans
orphans = dim_product_df.filter(pl.col("product_category_name_english").is_null())
null_keys = orphans.height

if null_keys > 0:
    logging.warning(f"Found {null_keys} orphan products without an English category translation! They will be loaded with NULL english names.")
    
    # Investigate Orphan Categories
    orphan_cats = orphans.select([
        "product_category_name"
    ]).unique().sort("product_category_name")
    
    print("\n--- ORPHAN INVESTIGATION (Category) ---")
    print(f"Total Unique Missing Categories: {orphan_cats.height}")
    print(orphan_cats.head(10))

# ETL Metrics
matched = dim_product_df.height - null_keys
success_rate = (matched / dim_product_df.height) * 100

print("\n--- ETL METRICS ---")
print(f"Rows processed:    {products.height:,}")
print(f"Rows loaded:       {dim_product_df.height:,}")
print(f"Translated categories: {matched:,}")
print(f"Missing translations:  {null_keys:,}")
print(f"Success rate:      {success_rate:.2f}%")

print("\nShape:", dim_product_df.shape)
logging.info("Product validation passed.")

2026-07-26 02:12:59,298 - INFO - Transforming Product Dimension...


2026-07-26 02:12:59,329 - WARNING - Found 623 orphan products without an English category translation! They will be loaded with NULL english names.


2026-07-26 02:12:59,333 - INFO - Product validation passed.



--- ORPHAN INVESTIGATION (Category) ---
Total Unique Missing Categories: 3
shape: (3, 1)
┌─────────────────────────────────┐
│ product_category_name           │
│ ---                             │
│ str                             │
╞═════════════════════════════════╡
│ null                            │
│ pc_gamer                        │
│ portateis_cozinha_e_preparador… │
└─────────────────────────────────┘

--- ETL METRICS ---
Rows processed:    32,951
Rows loaded:       32,951
Translated categories: 32,328
Missing translations:  623
Success rate:      98.11%

Shape: (32951, 11)
CPU times: total: 15.6 ms
Wall time: 36.7 ms


In [17]:
%%time
# LOAD
logging.info("Loading Product Dimension to MySQL...")

df_pandas = dim_product_df.to_pandas()

with engine.begin() as conn:
    # Use DELETE instead of TRUNCATE to respect foreign key constraints if dependent fact rows exist
    conn.execute(text("DELETE FROM dim_date;"))
    conn.execute(text("DELETE FROM dim_product;"))
    df_pandas.to_sql("dim_product", conn, if_exists="append", index=False)

logging.info(f"Loaded {len(df_pandas):,} records into dim_product.")

2026-07-26 02:12:59,344 - INFO - Loading Product Dimension to MySQL...


2026-07-26 02:13:15,588 - INFO - Loaded 32,951 records into dim_product.


CPU times: total: 891 ms
Wall time: 16.2 s


In [18]:
# VALIDATE (SQL)
with engine.connect() as conn:
    row_count = conn.execute(text("SELECT COUNT(*) FROM dim_product;")).scalar()
    print(f"Total Rows in MySQL dim_product: {row_count}")
    
    sample = conn.execute(text("SELECT product_id, product_category_name_english, product_volume_cm3 FROM dim_product LIMIT 3;")).fetchall()
    print("\nSample Database Records:")
    for row in sample:
        print(row)

Total Rows in MySQL dim_product: 32951

Sample Database Records:
('1e9e8ef04dbcff4541ed26657ea517e5', 'perfumery', 2240)
('3aa071139cb16b67ca9e5dea641aaa2f', 'art', 10800)
('96bd76ec8810374ed1b65e291975717f', 'sports_leisure', 2430)


## 9. Date Dimension ETL
**Transformations**:
- Extract `min` and `max` dates from the `orders` dataset.
- Generate a continuous calendar using `pl.date_range`.
- Compute all required date attributes (day, month, quarter, year, etc).


In [19]:
%%time
# TRANSFORM
logging.info("Transforming Date Dimension...")
from datetime import datetime, date

# 1. Get min and max dates from orders
# Cast strings to datetime
orders_dt = orders.with_columns([
    pl.col("order_purchase_timestamp").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False),
    pl.col("order_estimated_delivery_date").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S", strict=False)
])

min_date = orders_dt["order_purchase_timestamp"].min().date()
max_date = orders_dt["order_estimated_delivery_date"].max().date()

print(f"Generating calendar from {min_date} to {max_date}")

# 2. Generate Date Range
dates_series = pl.date_range(min_date, max_date, "1d", eager=True).alias("full_date")

# 3. Compute Attributes
dim_date_df = (
    pl.DataFrame([dates_series])
    .with_columns([
        pl.col("full_date").dt.strftime("%Y%m%d").cast(pl.Int32).alias("date_key"),
        pl.col("full_date").dt.day().cast(pl.UInt8).alias("day_number"),
        pl.col("full_date").dt.strftime("%A").alias("day_name"),
        pl.col("full_date").dt.week().cast(pl.UInt8).alias("week_number"),
        pl.col("full_date").dt.month().cast(pl.UInt8).alias("month_number"),
        pl.col("full_date").dt.strftime("%B").alias("month_name"),
        pl.col("full_date").dt.strftime("%b").alias("month_short_name"),
        pl.col("full_date").dt.strftime("%Y-%m").alias("month_year"),
        pl.col("full_date").dt.quarter().cast(pl.UInt8).alias("quarter_number"),
        ("Q" + pl.col("full_date").dt.quarter().cast(pl.String)).alias("quarter_name"),
        (pl.col("full_date").dt.year().cast(pl.String) + "Q" + pl.col("full_date").dt.quarter().cast(pl.String)).alias("quarter_label"),
        pl.col("full_date").dt.year().cast(pl.UInt16).alias("year_number"),
        pl.col("full_date").dt.ordinal_day().cast(pl.UInt16).alias("day_of_year"),
        (pl.col("full_date").dt.weekday() >= 6).alias("is_weekend"),
        (pl.col("full_date").dt.day() == 1).alias("is_month_start")
    ])
    .with_columns(
        (pl.col("full_date").dt.month() != (pl.col("full_date") + pl.duration(days=1)).dt.month()).alias("is_month_end")
    )
)

# VALIDATE (Polars)
assert dim_date_df["date_key"].n_unique() == dim_date_df.height, "Duplicate date_keys found!"

print("\n--- ETL METRICS ---")
print(f"Dates generated: {dim_date_df.height:,}")
print("\nSample:")
print(dim_date_df.head(3))

2026-07-26 02:13:15,735 - INFO - Transforming Date Dimension...


Generating calendar from 2016-09-04 to 2018-11-12

--- ETL METRICS ---
Dates generated: 800

Sample:
shape: (3, 17)
┌────────────┬──────────┬───────────┬──────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ full_date  ┆ date_key ┆ day_numbe ┆ day_name ┆ … ┆ day_of_ye ┆ is_weeken ┆ is_month_ ┆ is_month_ │
│ ---        ┆ ---      ┆ r         ┆ ---      ┆   ┆ ar        ┆ d         ┆ start     ┆ end       │
│ date       ┆ i32      ┆ ---       ┆ str      ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆          ┆ u8        ┆          ┆   ┆ u16       ┆ bool      ┆ bool      ┆ bool      │
╞════════════╪══════════╪═══════════╪══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 2016-09-04 ┆ 20160904 ┆ 4         ┆ Sunday   ┆ … ┆ 248       ┆ true      ┆ false     ┆ false     │
│ 2016-09-05 ┆ 20160905 ┆ 5         ┆ Monday   ┆ … ┆ 249       ┆ false     ┆ false     ┆ false     │
│ 2016-09-06 ┆ 20160906 ┆ 6         ┆ Tuesday  ┆ … ┆ 250       ┆ false     ┆

In [20]:
%%time
# LOAD
logging.info("Loading Date Dimension to MySQL...")

df_pandas = dim_date_df.to_pandas()

with engine.begin() as conn:
    conn.execute(text("DELETE FROM dim_date;"))
    df_pandas.to_sql("dim_date", conn, if_exists="append", index=False)

logging.info(f"Loaded {len(df_pandas):,} records into dim_date.")

2026-07-26 02:13:15,856 - INFO - Loading Date Dimension to MySQL...


2026-07-26 02:13:15,968 - INFO - Loaded 800 records into dim_date.


CPU times: total: 46.9 ms
Wall time: 114 ms


## 10. Final Warehouse Validation
We will now query the data warehouse to validate integrity across the entire dimensional model.


In [21]:
# WAREHOUSE VALIDATION
logging.info("Validating Data Warehouse Integrity...")

with engine.connect() as conn:
    print("============================")
    print("WAREHOUSE VALIDATION SUMMARY")
    print("============================")
    
    # Row Counts
    print("\n1. ROW COUNTS")
    tables = ['dim_geography', 'dim_customer', 'dim_seller', 'dim_product', 'dim_date']
    for t in tables:
        cnt = conn.execute(text(f"SELECT COUNT(*) FROM {t};")).scalar()
        print(f"{t.ljust(15)} : {cnt:,}")
        
    # Duplicate Keys
    print("\n2. BUSINESS KEY UNIQUENESS")
    keys = {'dim_customer': 'customer_id', 'dim_seller': 'seller_id', 'dim_product': 'product_id'}
    for t, k in keys.items():
        dupes = conn.execute(text(f"SELECT COUNT(*) FROM (SELECT {k} FROM {t} GROUP BY {k} HAVING COUNT(*) > 1) as sub;")).scalar()
        status = "PASS" if dupes == 0 else f"FAIL ({dupes} dupes)"
        print(f"{t.ljust(15)} : {status}")
        
    # Orphans
    print("\n3. ORPHAN RECORDS (NULL FKs)")
    cust_orphans = conn.execute(text("SELECT COUNT(*) FROM dim_customer WHERE geography_key IS NULL;")).scalar()
    seller_orphans = conn.execute(text("SELECT COUNT(*) FROM dim_seller WHERE geography_key IS NULL;")).scalar()
    prod_orphans = conn.execute(text("SELECT COUNT(*) FROM dim_product WHERE product_category_name_english IS NULL;")).scalar()
    
    print(f"Customer Orphans (Missing Geo) : {cust_orphans}")
    print(f"Seller Orphans (Missing Geo)   : {seller_orphans}")
    print(f"Product Orphans (Missing Cat)  : {prod_orphans}")

logging.info("Validation complete.")

2026-07-26 02:13:15,979 - INFO - Validating Data Warehouse Integrity...


WAREHOUSE VALIDATION SUMMARY

1. ROW COUNTS
dim_geography   : 19,015


dim_customer    : 99,441
dim_seller      : 3,095
dim_product     : 32,951
dim_date        : 800

2. BUSINESS KEY UNIQUENESS


dim_customer    : PASS
dim_seller      : PASS
dim_product     : PASS

3. ORPHAN RECORDS (NULL FKs)


2026-07-26 02:13:17,068 - INFO - Validation complete.


Customer Orphans (Missing Geo) : 278
Seller Orphans (Missing Geo)   : 7
Product Orphans (Missing Cat)  : 623


## 12. ETL Summary & Data Quality Report
**Data Quality Notes (Orphans):**
- **Customers & Sellers:** Missing `geography_key` mappings occur because the Olist dataset contains slightly varied spellings of city names and zip codes that did not perfectly map during our deduplication phase. These are expected data quality gaps from the source system.
- **Products:** Missing English translations occur because the provided translation CSV `product_category_name_translation.csv` does not have 100% coverage of all Portuguese categories found in the main `products` dataset.

Below we generate the final aggregated ETL Summary Report and export it for downstream documentation.

In [22]:
import pandas as pd
from IPython.display import display

# Constructing the Final ETL Summary Table
summary_data = {
    'Dimension': ['Geography', 'Customer', 'Seller', 'Product', 'Date'],
    'Loaded': [19015, 99441, 3095, 32951, 800],
    'Warnings (Orphans/Missing)': [0, 278, 7, 623, 0],
    'Status': ['✅', '✅', '✅', '✅', '✅']
}

summary_df = pd.DataFrame(summary_data)

# Display beautifully in notebook
print("=========================================")
print("          ETL PIPELINE SUMMARY           ")
print("=========================================")
display(summary_df)

# Export to CSV
summary_df.to_csv('etl_metrics.csv', index=False)
logging.info("Exported ETL Summary to etl_metrics.csv")

          ETL PIPELINE SUMMARY           


,Dimension,Loaded,Warnings (Orphans/Missing),Status
0,Geography,19015,0,✅
1,Customer,99441,278,✅
2,Seller,3095,7,✅
3,Product,32951,623,✅
4,Date,800,0,✅


2026-07-26 02:13:17,111 - INFO - Exported ETL Summary to etl_metrics.csv


## 11. Pipeline Completed Successfully! 🚀
All dimensions have been securely loaded into the Data Warehouse.
